# 05 — Sénat : socle depuis les données ouvertes

**Rôle :** constituer le socle Sénat depuis une source **ouverte et transparente** — Senate Stock Watcher (JSON sur GitHub). On utilise **`all_daily_summaries.json`** : il fournit, par dépôt, la **date de réception (= `disclosure_date`, anti look-ahead)** ET le **BioGuideID** du sénateur — deux champs absents de `all_transactions.json`.

**⚠️ Limite de fraîcheur (constatée à l'exécution) :** ce dataset ouvert s'arrête vers **mars 2021**. Pour la queue 2021→aujourd'hui, voir le secours **kadoa** (dernière cellule) ou la vérification eFD (notebook 06). Les lignes sans ticker (obligations/options) sont flaguées (**backlog**).

**Pas de scraping d'efdsearch ici** (Akamai + gate d'agrément) : on ne contourne aucune protection.

**Cellule 1 — Imports & chemins.**

In [1]:
import requests, json
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
EXT_SENATE   = PROJECT_ROOT / "data" / "external" / "senate_openset"; EXT_SENATE.mkdir(parents=True, exist_ok=True)
PROC_SENATE  = PROJECT_ROOT / "data" / "processed" / "senate"; PROC_SENATE.mkdir(parents=True, exist_ok=True)
USER_AGENT   = "Ramify-QIS research (contact: <ton-email>)" 

**Cellule 2 — Téléchargement du dataset ouvert.** `all_daily_summaries.json` = résumés par dépôt, avec `date_recieved` et `bioguide`. Idempotent.

In [2]:
SSW_URL = ("https://raw.githubusercontent.com/timothycarambat/"
           "senate-stock-watcher-data/master/aggregate/all_daily_summaries.json")
raw_path = EXT_SENATE / "ssw_all_daily_summaries.json"
if not raw_path.exists():
    r = requests.get(SSW_URL, headers={"User-Agent": USER_AGENT}, timeout=120)
    r.raise_for_status()
    raw_path.write_bytes(r.content)
summaries = json.loads(raw_path.read_text())
print("Dépôts (daily summaries) :", len(summaries))

Dépôts (daily summaries) : 1442


**Cellule 3 — Aperçu d'un dépôt.** Un sénateur + `date_recieved` + une liste de transactions.

In [3]:
{k: (f"<{len(v)} transactions>" if k == "transactions" else v) for k, v in summaries[0].items()}

{'first_name': 'Susan M',
 'last_name': 'Collins',
 'office': 'Collins, Susan M. (Senator)',
 'ptr_link': 'https://efdsearch.senate.gov/search/view/ptr/32550a8f-923e-416f-84f3-e19ab4f148b1/',
 'date_recieved': '03/10/2021',
 'transactions': '<2 transactions>',
 'bioguide': 'C001035'}

**Cellule 4 — Aplatissement → une ligne par transaction.** On propage `date_recieved` (disclosure) et `bioguide` à chaque transaction.

In [4]:
rows = []
for s in summaries:
    base = {"source": "senate", "provenance": "openset",
            "bioguide_id": s.get("bioguide"),
            "declarant_name": f'{s.get("first_name","")} {s.get("last_name","")}'.strip(),
            "disclosure_date": s.get("date_recieved"),
            "source_url": s.get("ptr_link")}
    for t in s.get("transactions", []):
        rows.append({**base,
            "transaction_date": t.get("transaction_date"), "owner": t.get("owner"),
            "ticker": t.get("ticker"), "asset_description": t.get("asset_description"),
            "asset_type": t.get("asset_type"), "operation_type": t.get("type"),
            "amount_range": t.get("amount")})
senate = pd.DataFrame(rows)
senate["ticker"] = senate["ticker"].replace({"--": None})
print("Transactions :", len(senate), "| sans ticker :", int(senate["ticker"].isna().sum()),
      "| avec BioGuideID :", int(senate["bioguide_id"].notna().sum()))

Transactions : 8030 | sans ticker : 1667 | avec BioGuideID : 8030


**Cellule 5 — Sauvegarde + couverture par an (disclosure).** Met en évidence la frontière de fraîcheur (~2021).

In [5]:
senate["disclosure_date"] = pd.to_datetime(senate["disclosure_date"], errors="coerce")
senate.to_csv(PROC_SENATE / "senate_transactions.csv", index=False)
print(senate["disclosure_date"].dt.year.value_counts().sort_index())

disclosure_date
2014     647
2015    1104
2016     967
2017    1314
2018    1205
2019    1124
2020    1574
2021      95
Name: count, dtype: int64


**Cellule 6 — (Recommandé) Secours kadoa pour la queue récente.** Le socle s'arrêtant ~2021, complétez 2021→aujourd'hui avec le dataset MIT kadoa (« 2012→présent ») : renseignez l'URL du JSON et concaténez (même schéma, `provenance='openset-kadoa'`).

In [6]:
# Cellule 6 — Queue Sénat 2021→présent via kadoa (MIT, mise à jour quotidienne).
# Source : kadoa-org/congress-trading-monitor — fichiers par filer dans public/data/filer/.
# BioGuideID extrait de photo_url (ex: ".../B001236.jpg" → "B001236").
# disclosure_date = filing_date (date eFD réelle, anti look-ahead).
# On ne prend que les transactions dont filing_date >= 2021-01-01 pour splicer sur le socle SSW.

KADOA_BASE = "https://raw.githubusercontent.com/kadoa-org/congress-trading-monitor/main/public/data"
KADOA_CUTOFF = "2021-01-01"

# 1 — Liste des filers Sénat
filers_path = EXT_SENATE / "kadoa_filers.json"
if not filers_path.exists():
    r = requests.get(f"{KADOA_BASE}/filers.json", headers={"User-Agent": USER_AGENT}, timeout=60)
    r.raise_for_status(); filers_path.write_bytes(r.content)
filers_all = json.loads(filers_path.read_text())
senate_filers = [f for f in filers_all if f.get("chamber") == "senate"]
print(f"Filers Sénat kadoa : {len(senate_filers)}")

# 2 — Téléchargement des fichiers par filer + extraction des transactions
from pathlib import Path as _P
kadoa_rows = []
for filer in senate_filers:
    fid = filer["id"]
    fp = EXT_SENATE / f"kadoa_{fid}.json"
    if not fp.exists():
        try:
            r = requests.get(f"{KADOA_BASE}/filer/{fid}.json",
                             headers={"User-Agent": USER_AGENT}, timeout=60)
            r.raise_for_status(); fp.write_bytes(r.content)
        except Exception as e:
            print(f"  SKIP {fid}: {e}"); continue
    fdata = json.loads(fp.read_text())
    
    # BioGuideID depuis photo_url
    photo = filer.get("photo_url", "")
    bioguide = _P(photo).stem if photo else None
    
    for t in fdata.get("trades", []):
        fd = t.get("filing_date") or ""
        if fd < KADOA_CUTOFF:
            continue
        amount_lo = t.get("amount_range_low")
        amount_hi = t.get("amount_range_high")
        midpoint  = ((amount_lo or 0) + (amount_hi or 0)) / 2 if (amount_lo or amount_hi) else None
        kadoa_rows.append({
            "source":           "senate",
            "provenance":       "openset-kadoa",
            "bioguide_id":      bioguide,
            "declarant_name":   filer.get("full_name"),
            "disclosure_date":  fd,
            "source_url":       t.get("doc_url"),
            "transaction_date": t.get("transaction_date"),
            "owner":            t.get("owner"),
            "ticker":           t.get("ticker"),
            "asset_description":t.get("asset_name"),
            "asset_type":       t.get("asset_type"),
            "operation_type":   t.get("transaction_type"),
            "amount_range":     t.get("amount_range_label"),
            "amount_midpoint":  midpoint,
        })

kadoa_df = pd.DataFrame(kadoa_rows)
print(f"Transactions kadoa (>= {KADOA_CUTOFF}) : {len(kadoa_df)}")
if len(kadoa_df):
    print("Couverture :", kadoa_df["disclosure_date"].min(), "→", kadoa_df["disclosure_date"].max())

# 3 — Concat socle SSW + queue kadoa + sauvegarde
senate_full = pd.concat([senate, kadoa_df], ignore_index=True)
senate_full["disclosure_date"] = pd.to_datetime(senate_full["disclosure_date"], errors="coerce")
senate_full.to_csv(PROC_SENATE / "senate_transactions.csv", index=False)
print("\nCouverture finale (disclosure_date par an) :")
print(senate_full["disclosure_date"].dt.year.value_counts().sort_index())
print("\nTotal Sénat :", len(senate_full), "| Sans ticker :", int(senate_full["ticker"].isna().sum()))

Filers Sénat kadoa : 60


Transactions kadoa (>= 2021-01-01) : 4233
Couverture : 2021-01-05 → 2026-06-16

Couverture finale (disclosure_date par an) :
disclosure_date
2014     647
2015    1104
2016     967
2017    1314
2018    1205
2019    1124
2020    1574
2021     528
2022     639
2023     922
2024     780
2025     964
2026     495
Name: count, dtype: int64

Total Sénat : 12263 | Sans ticker : 2798


Socle Sénat prêt ✅ (avec `disclosure_date` + BioGuideID) — vérification de provenance au **`06_senate_efd_provenance`**.